In [ ]:
# 05_pseudobulk_de.ipynb
# this notebook is to compute pseudobulk, as a parallel method to Wilcoxon ranked (Mann-Whitney U) DE
# pseudobulk is not integrated in scanpy, the code here is custom

import scanpy as sc
import pandas as pd
import numpy as np
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

adata = sc.read_h5ad("../data/interim/04_adata_responder_flagged.h5ad")
wilcoxon_de = pd.read_csv("../data/interim/04_wilcoxon_de_results.csv.gz")
print(adata)
print(wilcoxon_de.shape)

In [ ]:
# custom fxn to do pseudobulk
# gemgroup is the lane
# label is a string we pass to name the group i.e. "KLF1" or "control"

# tldr, this fxn returns a list of dicts, where each is a pseudo-replicate "sample"
# defined by the intersectino of "label" (i.e. KLF1 or ctrl) and gemgroup (replicate number i.e. lane number)
# calling this fxn twice (once for GOI and once for ctrl) and concat the two lists gives full (target + ctrl) pseudobulk datasets

def pseudobulk_group(adata, cell_mask, label):
    """Sum raw counts across cells in cell_mask, per gemgroup, for one target."""
    # subset for data from "target". cell_mask telling which cells (rows) are target, needs to be created before running this fxn.
    sub = adata[cell_mask]
    rows = []
    for gg in sorted(sub.obs["gemgroup"].unique()):
        # creates a mask defining "replicates" (lanes)
        gg_mask = (sub.obs["gemgroup"] == gg).values
        if gg_mask.sum() < 10:  # skip pseudo-replicates with <10 cells, too few cells to be meaningful
            continue
        # using counts layer; raw data.  .sum(axis=0) sums down each column (creates pseudobulk count value per replicate)
        counts_sum = np.asarray(sub.layers["counts"][gg_mask].sum(axis=0)).flatten()
        rows.append({"sample": f"{label}_gg{gg}", "condition": label, "gemgroup": gg,
                      **dict(zip(sub.var_names, counts_sum))})
    return rows

In [ ]:
# benchmark on KLF1 before committing to full 105-target loop

is_control = (adata.obs["target"] == "control").values
# note the & here: KLF1 perturbed cells. is_responder is whether the perturb increased KLF levels (only take truly perturbed cells)
is_klf1 = (adata.obs["target"] == "KLF1").values & (adata.obs["is_responder"]).values

rows = pseudobulk_group(adata, is_klf1, "KLF1") + pseudobulk_group(adata, is_control, "control")
pb_df = pd.DataFrame(rows).set_index("sample")

counts_matrix = pb_df.drop(columns=["condition", "gemgroup"])
meta = pb_df[["condition", "gemgroup"]]

print(counts_matrix.shape, meta.shape)
meta

In [ ]:
# still testing: run the actual DE for KLF1 examples
 
import time

start = time.time()

dds = DeseqDataSet(
    counts=counts_matrix.astype(int),
    metadata=meta,
    design="~condition",
    refit_cooks=True,
)
dds.deseq2()

stat_res = DeseqStats(dds, contrast=["condition", "KLF1", "control"])
stat_res.summary()

elapsed = time.time() - start
print(f"elapsed: {elapsed:.1f}s")

stat_res.results_df.sort_values("padj", ascending=True).head(10)

In [ ]:
# these are sorted by ascending padj.  this is the right way to look at "top DEGs"
# this list is similar to the one generated by Wilcoxon.

In [ ]:
# run psuedobulk DE on the full 105 (perturbed) gene set

import time
import io
from contextlib import redirect_stdout

single_targets = adata.obs.loc[adata.obs["perturbation_type"] == "single", "target"].unique()
is_control = (adata.obs["target"] == "control").values
control_rows = pseudobulk_group(adata, is_control, "control")  # same for every comparison, compute once

all_results = []
failed = []
start = time.time()

for i, gene in enumerate(single_targets):
    if gene not in adata.var_names:
        continue
    is_target = (adata.obs["target"] == gene).values & (adata.obs["is_responder"]).values

    target_rows = pseudobulk_group(adata, is_target, gene)
    if len(target_rows) < 2:  # need at least 2 pseudo-replicates per condition for DESeq2
        failed.append((gene, "too few replicates"))
        continue

    pb_df = pd.DataFrame(target_rows + control_rows).set_index("sample")
    counts_matrix = pb_df.drop(columns=["condition", "gemgroup"]).astype(int)
    meta = pb_df[["condition", "gemgroup"]]

    try:
        with redirect_stdout(io.StringIO()):  # suppress PyDESeq2's verbose per-gene logging
            dds = DeseqDataSet(counts=counts_matrix, metadata=meta, design="~condition")
            dds.deseq2()
            stat_res = DeseqStats(dds, contrast=["condition", gene, "control"])
            stat_res.summary()
        res = stat_res.results_df.dropna().reset_index()
        res = res.rename(columns={res.columns[0]: "gene_name"})
        res["target"] = gene
        all_results.append(res)
    except Exception as e:
        failed.append((gene, str(e)))

    if (i + 1) % 15 == 0:
        print(f"{i+1}/{len(single_targets)} done, {time.time()-start:.0f}s elapsed")

pseudobulk_de = pd.concat(all_results, ignore_index=True)
print(f"\nDone: {pseudobulk_de.shape}, {len(failed)} failed, {time.time()-start:.0f}s total")
if failed:
    print(failed[:10])

In [ ]:
# save the results to csv
pseudobulk_de.to_csv("../data/interim/05_pseudobulk_de_results.csv.gz", index=False)

In [ ]:
# no need to save adata in this notebook, it was not modified. we saved DE results separately in the CSV above